### 环境设置

In [ ]:
import os

# 设置你的 API 密钥 (Bearer <token> 中的 token)
# os.environ["OPENAI_API_KEY"] = "your_api_key_here"
# os.environ["OPENAI_API_BASE"] = "your_api_base_here"
# os.environ["TAVILY_API_KEY"] = "your_tavily_api_key_here"

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


sk-yuknxsbirvgfjucumekpjaytgbgsvgvgdyztihhcqmtwlafu


不使用 langchain 包，直接调用 OpenAI API:

In [2]:
from openai import OpenAI

client = OpenAI(
    # api_key='your_api_key_here', # 替换自己的api_key
    # base_url="your_api_base_here"  # 替换自己的api_base,
)

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-32B-Instruct", 
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant talented at cooking skill.'},
        {'role': 'user', 'content': '告诉我如何做地三鲜'}
    ]
)
print(completion.choices[0].message.content)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-yuknx***************************************lafu. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

使用 langchain 进行简化：

In [3]:
from langchain_openai import ChatOpenAI

user_message = [
    SystemMessage(content='你是一个说话阴阳怪气的助手'),
    HumanMessage(content="你是谁？想从我这儿偷东西吗？还不赶快把我的钱还给我！")
]
buddy = ChatOpenAI(model="Qwen/Qwen2.5-32B-Instruct")
response = buddy.invoke(user_message)
print(response.content, "\n")
for a, b in response:
    print(f"key = {a};\nval = {b}")

哦哟，看来你误会大了，我这人最讨厌偷东西了，不仅不偷，看到别人偷还会上前大喊一声“喂，这样多没品啊”。放心吧，你的钱包在你身上，稳如泰山，我可没那只手。咱俩好好交流，保证不谈偷鸡摸狗的事儿。有啥我能真正帮到你的？ 

key = content;
val = 哦哟，看来你误会大了，我这人最讨厌偷东西了，不仅不偷，看到别人偷还会上前大喊一声“喂，这样多没品啊”。放心吧，你的钱包在你身上，稳如泰山，我可没那只手。咱俩好好交流，保证不谈偷鸡摸狗的事儿。有啥我能真正帮到你的？
key = additional_kwargs;
val = {'refusal': None}
key = response_metadata;
val = {'token_usage': {'completion_tokens': 82, 'prompt_tokens': 40, 'total_tokens': 122, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen2.5-32B-Instruct', 'system_fingerprint': '', 'id': '019b454ee082300826f5059f9a32bede', 'finish_reason': 'stop', 'logprobs': None}
key = type;
val = ai
key = name;
val = None
key = id;
val = lc_run--019b454e-d75b-7053-947d-44367e089c1c-0
key = tool_calls;
val = []
key = invalid_tool_calls;
val = []
key = usage_metadata;
val = {'input_tokens': 40, 'output_tokens': 82, 'total_tokens': 122, 'input_token_details': {}, 'output_token_details': {}}


加入 langchain 提示词 `ChatPromptTemplate`

In [66]:
from langchain_core.prompts import ChatPromptTemplate

template_string = """将以下用三重反引号括起来的文本翻译成风格为{style}的文本，只需要输出文本即可。\\n文本: ```{text}```"""

# 创建 ChatPromptTemplate 实例
prompt_template = ChatPromptTemplate.from_template(template_string)

# 查看模版中的输入变量
print(prompt_template.messages[0].prompt.input_variables)  # 输出: ['style', 'text']


['style', 'text']


In [69]:
# 定义客户的投诉风格和文本
customer_style = """语气平和且尊重"""
customer_email = """
啊，我的搅拌机盖子飞了出去，把我的厨房墙壁弄得满是奶昔！更糟糕的是，保修不包括清理厨房的费用。我现在需要你的帮助，伙计！
"""

# 使用模版格式化消息
customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)

# 打印生成的消息类型和内容
print(type(customer_messages))         # 输出: <class 'list'>
print(type(customer_messages[0]))      # 输出: <class 'langchain.schema.HumanMessage'>
print(customer_messages)

user_message = [
    SystemMessage(content='你是一个友好的助手'),
    customer_messages[0]
]

print(buddy.invoke(customer_messages).content)

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>
[HumanMessage(content='将以下用三重反引号括起来的文本翻译成风格为语气平和且尊重的文本，只需要输出文本即可。\\n文本: ```\n啊，我的搅拌机盖子飞了出去，把我的厨房墙壁弄得满是奶昔！更糟糕的是，保修不包括清理厨房的费用。我现在需要你的帮助，伙计！\n```', additional_kwargs={}, response_metadata={})]
我的搅拌机盖子不小心飞出去了，导致厨房的墙面上沾满了奶昔。而且，保修服务并不涵盖清洁费用。现在我需要帮助来解决这个问题。谢谢。


In [131]:
# Docs 示例
system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = buddy.invoke(messages)
print(response.content)

Creating a REST API involves defining endpoints where data can be accessed or modified using standard HTTP methods like GET, POST, PUT, and DELETE. REST stands for Representational State Transfer and is an architectural style for designing distributed systems, like web applications and services.

### Step-by-Step Guide Using Flask

Flask is a minimalistic web framework for Python that can be used to create a REST API easily. Below is a simple example of creating a REST API using Flask:

1. **Install Flask**: First, if Flask is not already installed, you can install it using pip:
   ```bash
   pip install Flask
   ```

2. **Basic API Example**:
   Suppose we create a simple API to manage a list of books. We will create endpoints to get all books, add a new book, get a book by ID, update a book, and delete a book.

   ```python
   from flask import Flask, jsonify, request

   app = Flask(__name__)

   # In-memory database simulation
   books = [
       {'id': 1, 'title': '1984', 'author'

### 结构化输出
两种结构化输出，后者使用了 langchain 提供的工具包。

In [132]:
review_template = """
请从以下文本中提取以下信息：

1. 是否作为礼物购买（gift）：如果是礼物，请回答True；如果不是或未知，请回答False。
2. 到货天数（delivery_days）：提取产品到货所需的天数，如果信息不存在，请输出-1。
3. 价格或价值的评价（price_value）：提取任何与价格或价值相关的句子，并以逗号分隔的Python列表形式输出。

请以JSON格式输出，键包括：
- gift
- delivery_days
- price_value

文本：{text}
"""
# 创建 ChatPromptTemplate 实例
prompt_template = ChatPromptTemplate.from_template(review_template)

# 示例用户评论
customer_review = """
这款叶吹机非常棒！它有四个设置：蜡烛吹风、微风、风城和龙卷风模式。
它送得挺快的，三天就到了，刚好赶上我妻子的周年纪念日。
我觉得它稍微比其他叶吹机贵一些，但多出来的功能让我觉得物有所值。
"""

# 格式化提示
messages = prompt_template.format_messages(text=customer_review)
print(messages)
print("——————————")
response = buddy.invoke(messages)
print(response.content)

[HumanMessage(content='\n请从以下文本中提取以下信息：\n\n1. 是否作为礼物购买（gift）：如果是礼物，请回答True；如果不是或未知，请回答False。\n2. 到货天数（delivery_days）：提取产品到货所需的天数，如果信息不存在，请输出-1。\n3. 价格或价值的评价（price_value）：提取任何与价格或价值相关的句子，并以逗号分隔的Python列表形式输出。\n\n请以JSON格式输出，键包括：\n- gift\n- delivery_days\n- price_value\n\n文本：\n这款叶吹机非常棒！它有四个设置：蜡烛吹风、微风、风城和龙卷风模式。\n它送得挺快的，三天就到了，刚好赶上我妻子的周年纪念日。\n我觉得它稍微比其他叶吹机贵一些，但多出来的功能让我觉得物有所值。\n\n', additional_kwargs={}, response_metadata={})]
——————————
```json
{
  "gift": true,
  "delivery_days": 3,
  "price_value": [
    "我觉得它稍微比其他叶吹机贵一些，但多出来的功能让我觉得物有所值。"
  ]
}
```


In [127]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

# 定义每个字段的输出模式
gift_schema = ResponseSchema(name="gift", description="是否作为礼物购买", type="bool")
delivery_days_schema = ResponseSchema(name="delivery_days", description="产品到货天数", type="int")
price_value_schema = ResponseSchema(name="price_value", description="价格或价值的评价")

# 将所有模式组合成一个列表
response_schemas = [gift_schema, delivery_days_schema, price_value_schema]

# 使用模式列表初始化输出解析器
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = output_parser.get_format_instructions()

print(output_parser)
print("<output parser —————————— format instructions>")
print(format_instructions)

# 将解析指令嵌入提示模版
review_template_with_format = """
请从以下文本中提取以下信息：

1. 是否作为礼物购买（gift）：如果是礼物，请回答True；如果不是或未知，请回答False。
2. 到货天数（delivery_days）：提取产品到货所需的天数，如果信息不存在，请输出-1。
3. 价格或价值的评价（price_value）：提取任何与价格或价值相关的句子，并以逗号分隔的Python列表形式输出。

{format_instructions}

文本：{text}
"""

prompt_template = ChatPromptTemplate.from_template(review_template_with_format)
customer_review = """
这款叶吹机非常棒！它有四个设置：蜡烛吹风、微风、风城和龙卷风模式。
它送得挺快的，三天就到了，刚好赶上我妻子的周年纪念日。
我觉得它稍微比其他叶吹机贵一些，但多出来的功能让我觉得物有所值。
"""
messages = prompt_template.format_messages(text=customer_review, format_instructions=format_instructions)

response = buddy.invoke(messages)
print(response.content)
print("——————————")

output_dict = output_parser.parse(response.content)
print(output_dict)
print(output_dict.get('delivery_days')) # 输出 2

response_schemas=[ResponseSchema(name='gift', description='是否作为礼物购买', type='bool'), ResponseSchema(name='delivery_days', description='产品到货天数', type='int'), ResponseSchema(name='price_value', description='价格或价值的评价', type='string')]
<output parser —————————— format instructions>
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"gift": bool  // 是否作为礼物购买
	"delivery_days": int  // 产品到货天数
	"price_value": string  // 价格或价值的评价
}
```
```json
{
	"gift": true,
	"delivery_days": 3,
	"price_value": "我觉得它稍微比其他叶吹机贵一些，但多出来的功能让我觉得物有所值。"
}
```
——————————
{'gift': True, 'delivery_days': 3, 'price_value': '我觉得它稍微比其他叶吹机贵一些，但多出来的功能让我觉得物有所值。'}
3
